# 🧠 Entregable 3 — Análisis de Sentimientos en Tweets (3 Clases)

**Modelo:** Logistic Regression  
**Clases:** `0 = negativo`, `2 = neutral`, `4 = positivo`

En este trabajo se utiliza el famoso dataset de Twitter provisto por la cátedra para:

1. Limpiar y preprocesar los tweets (en inglés).
2. Construir un conjunto de entrenamiento con tres clases de sentimiento.
3. Vectorizar el texto con **TF-IDF**.
4. Entrenar un modelo de **Logistic Regression**.
5. Evaluar el rendimiento sobre el dataset de test manual (que incluye 0, 2 y 4).
6. Obtener métricas y conclusiones sobre la capacidad del modelo para distinguir los tres tipos de sentimiento.


## 1. Importación de librerías y configuración

In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

%matplotlib inline

## 2. Carga de datasets


Se utilizan los dos archivos provistos:

- `training.1600000.processed.noemoticon.csv` — 1.6M tweets con etiquetas 0 (neg), 4 (pos).
- `testdata.manual.2009.06.14.csv` — tweets etiquetados manualmente con 0 (neg), 2 (neutral), 4 (pos).

Para trabajar de forma razonable en el entorno de ejecución, se toma una **muestra** del dataset de entrenamiento.


In [ ]:
train_path = "/mnt/data/training.1600000.processed.noemoticon.csv"
test_path = "/mnt/data/testdata.manual.2009.06.14.csv"

cols = ["sentiment", "id", "date", "query", "user", "text"]

# Cargamos una muestra del dataset de entrenamiento para hacerlo manejable en clase
train_df = pd.read_csv(train_path, encoding="latin-1", names=cols, nrows=200000)
test_df = pd.read_csv(test_path, encoding="latin-1", names=cols)

train_df.head()

## 3. Preprocesamiento de texto (inglés)


Los tweets están en **inglés**, por lo tanto el preprocesamiento debe estar adaptado a este idioma.

Se realizan los siguientes pasos:

- Pasar todo a minúsculas.
- Eliminar menciones (`@usuario`).
- Eliminar URLs.
- Conservar solo palabras en inglés y **contracciones** (can't, don't, i'm, it's…).  
  Para esto se usa el patrón: `"[a-z]+(?:'[a-z]+)?"`.

Esto evita errores como convertir `can't` en `can t`, que generaría un token `can` y un token `t` sin sentido.


In [ ]:
def limpiar_tweet(t):
    t = str(t).lower()
    
    # Eliminar menciones y URLs
    t = re.sub(r'@\w+', ' ', t)
    t = re.sub(r'http\S+|www\S+', ' ', t)
    
    # Mantener solo palabras y CONTRACCIONES en inglés (can't, don't, i'm, etc.)
    tokens = re.findall(r"[a-z]+(?:'[a-z]+)?", t)
    
    return " ".join(tokens)

train_df["clean"] = train_df["text"].apply(limpiar_tweet)
test_df["clean"] = test_df["text"].apply(limpiar_tweet)

train_df[["text", "clean"]].head()

## 4. Preparación de las clases (0, 2, 4)


El dataset de entrenamiento solo trae clases **0** (negativo) y **4** (positivo).  
La consigna pide trabajar con **tres clases**: negativa, neutral y positiva.

Siguiendo el criterio discutido en clase, la clase *neutral* se construye de forma **subjetiva**:

- Negativo: tweets con etiqueta original `0`.
- Positivo: tweets con etiqueta original `4`.
- Neutral: subconjunto de tweets que **no contienen palabras emocionales fuertes**, ni claramente positivas ni claramente negativas (good, bad, love, hate, happy, sad, excellent, terrible, etc.).

Luego se vuelve a etiquetar este subconjunto como `2`.


In [ ]:
# Cantidad deseada por clase (si es posible)
N = 20000

# Muestras balanceadas de negativos y positivos
neg = train_df[train_df["sentiment"] == 0].sample(N, random_state=42)
pos = train_df[train_df["sentiment"] == 4].sample(N, random_state=42)

# Candidatos a neutrales: tweets sin palabras claramente emocionales
pattern_emociones = r"good|bad|love|hate|happy|sad|excellent|terrible|awesome|awful|amazing|horrible"
mask_neutral = ~train_df["clean"].str.contains(pattern_emociones, regex=True, na=False)

neutral_candidates = train_df[mask_neutral]

print("Cantidad de candidatos neutrales disponibles:", len(neutral_candidates))

# Tomamos hasta N neutrales (o menos, si no hay suficientes)
N_neutral = min(N, len(neutral_candidates))
neutral = neutral_candidates.sample(N_neutral, random_state=42).copy()
neutral.loc[:, "sentiment"] = 2  # reclasificamos como neutral

balanced_df = pd.concat([neg, neutral, pos], ignore_index=True)
balanced_df["sentiment"].value_counts()

## 5. Vectorización con TF-IDF


Se utiliza **TF-IDF (Term Frequency – Inverse Document Frequency)** para transformar cada tweet en un vector numérico:

- TF mide cuántas veces aparece cada término en el documento.
- IDF penaliza palabras muy frecuentes en todos los documentos.

Se limita el tamaño del vocabulario con `max_features` para mantener el modelo manejable.


In [ ]:
vectorizer = TfidfVectorizer(max_features=50000)

X_train = vectorizer.fit_transform(balanced_df["clean"])
y_train = balanced_df["sentiment"]

X_test = vectorizer.transform(test_df["clean"])
y_test = test_df["sentiment"]

X_train.shape, X_test.shape

## 6. Entrenamiento del modelo Logistic Regression


Se entrena un modelo de **Logistic Regression multinomial**, adecuado para clasificación multiclase (0, 2, 4).  
Este modelo fue visto teóricamente en la materia, por lo que está alineado con la consigna.


In [ ]:
model = LogisticRegression(
    max_iter=200,
    multi_class='multinomial',
    solver='lbfgs',
    n_jobs=-1
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Reporte de clasificación (0 = negativo, 2 = neutral, 4 = positivo):\n")
print(classification_report(y_test, y_pred))

## 7. Matriz de confusión


La matriz de confusión permite analizar dónde se equivoca más el modelo:

- Diagonal principal: predicciones correctas.
- Fuera de la diagonal: confusiones entre clases (por ejemplo, neutrales clasificados como positivos).


In [ ]:
labels = [0, 2, 4]
cm = confusion_matrix(y_test, y_pred, labels=labels)
cm

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap='Blues')

ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels)
ax.set_yticklabels(labels)
ax.set_xlabel('Predicción')
ax.set_ylabel('Real')
ax.set_title('Matriz de confusión — Logistic Regression (3 clases)')

for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, cm[i, j], ha='center', va='center', color='black')

plt.tight_layout()
plt.show()


## 8. Conclusiones

En este entregable se construyó un flujo completo y coherente de **análisis de sentimientos** con tres clases:

1. **Problema**: clasificar tweets como negativos, neutrales o positivos.
2. **Datos**: se utilizaron los datasets de Twitter provistos por la cátedra.
3. **Preprocesamiento**: normalización a minúsculas, eliminación de menciones y URLs, preservando contracciones en inglés para no destruir información semántica.
4. **Construcción de la clase neutral (2)**: siguiendo el criterio subjetivo discutido en clase, se seleccionaron tweets que no contienen palabras fuertemente emocionales como candidatos a neutrales.
5. **Vectorización**: se aplicó TF-IDF para representar cada tweet como un vector numérico.
6. **Modelo**: se entrenó una **Logistic Regression multinomial**, técnica vista en la materia.
7. **Evaluación**: se usó el dataset manual (0, 2, 4) para evaluar el modelo; se presentaron métricas de precisión, recall, F1-score y la matriz de confusión.

Este enfoque cumple con la consigna de:

- Realizar un análisis de sentimientos.
- Entrenar un modelo que prediga **los tres tipos de sentimiento**.
- Mantener coherencia metodológica desde la carga de datos hasta la interpretación de resultados.

En futuros trabajos se podría:

- Aumentar el tamaño del set de entrenamiento.
- Ajustar mejor la definición de "neutral".
- Probar otros modelos (SVM, redes neuronales) o representaciones más avanzadas (word embeddings, transformers), siempre que la materia lo permita.
